# AST Transfer Learning

This notebook implements the transfer learning pipeline using the Audio Spectrogram Transformer (AST) architecture on the GUITAR-FX dataset.

In [1]:
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Ensure project root is in sys.path
sys.path.append('..')
from src.data.dataset_ast import FxDatasetAST
from src.data.datasplit import DataSplit
from src.models.ast_models import ASTModel
from src.utils.device import get_device

%matplotlib inline

## 1. Configuration
Select the dataset scenario (mono/poly - disc/cont) and define transfer learning hyperparameters.

In [2]:
# --- SCENARIO SELECTOR ---
# 'mono_disc' | 'mono_cont' | 'poly_disc' | 'poly_cont'
DATASET_TYPE = 'mono_disc'

# Global hyperparameters
SEED = 42
EPOCHS_PROBE = 10     # Epochs for training only the linear classifier head
BATCH_SIZE = 4       # AST consumes more vRAM, use a smaller batch size
LR_PROBE = 1e-3
DEVICE_NAME = None
NUM_WORKERS = 0
CACHE_IN_RAM = True
TARGET_LENGTH = 1024  # AST expected input frames (adjust based on audio length)

# Scenario-specific dataset root and parameters
if DATASET_TYPE == 'mono_disc':
    DATASET_ROOT = '../datasets/GUITAR-FX/Mono_Discrete'
    EXCL_FOLDERS = ['_NoFX_mono', '_NoFX_mono_preprocessed', 'MT2']
elif DATASET_TYPE == 'mono_cont':
    DATASET_ROOT = '../../datasets/GUITAR-FX-DIST/Mono_Continuous'
    EXCL_FOLDERS = ['_NoFX_mono', '_NoFX_mono_preprocessed', 'MT2']
elif DATASET_TYPE == 'poly_disc':
    DATASET_ROOT = '../../datasets/GUITAR-FX-DIST/Poly_Discrete'
    EXCL_FOLDERS = ['_NoFX_mono', '_NoFX_mono_preprocessed', 'MT2']
elif DATASET_TYPE == 'poly_cont':
    DATASET_ROOT = '../../datasets/GUITAR-FX-DIST/Poly_Continuous'
    EXCL_FOLDERS = ['_NoFX_mono', '_NoFX_mono_preprocessed', 'MT2']
else:
    raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}")

SPECTRA_FOLDER = 'mel_16'

# Output directories
CHECKPOINTS_DIR = "../checkpoints"
RESULTS_DIR = "../results"
SCENARIO_RESULTS_DIR = os.path.join(RESULTS_DIR, f"ast_{DATASET_TYPE}")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SCENARIO_RESULTS_DIR, exist_ok=True)

print(f"[{DATASET_TYPE.upper()}] Configuration loaded.")

[MONO_DISC] Configuration loaded.


## 2. Data Loading
Uses the `FxDatasetAST` class optimized for the AST log-mel standard format.

In [3]:
device = get_device(DEVICE_NAME)
print(f"Selected compute device: {device}")

print("Loading dataset...")
dataset = FxDatasetAST(
    root=DATASET_ROOT,
    excl_folders=EXCL_FOLDERS,
    spectra_folder=SPECTRA_FOLDER,
    cache_in_ram=CACHE_IN_RAM,
    target_length=TARGET_LENGTH
)
dataset.init_dataset()

print(f"Total dataset samples: {len(dataset):,}")
print(f"Number of effect classes: {dataset.num_fx}")

split = DataSplit(
    dataset,
    test_train_split=0.8,
    val_train_split=0.1,
    shuffle=True,
    seed=SEED,
)

train_loader, val_loader, test_loader = split.get_split(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

Selected compute device: privateuseone:0
Loading dataset...
Total dataset samples: 123,552
Number of effect classes: 13


## 3. Model Setup
Loads the pre-trained AST model and sets up the loss criterion.

In [4]:
print("Loading AST Model pre-trained on ImageNet and AudioSet...")
model = ASTModel(
    label_dim=dataset.num_fx,
    input_tdim=TARGET_LENGTH,
    imagenet_pretrain=True,
    audioset_pretrain=True
).to(device)

criterion = nn.CrossEntropyLoss()

Loading AST Model pre-trained on ImageNet and AudioSet...
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True


c:\Users\caduo\dev\tcc\tcc\notebooks\..\src\models\ast_models.py:134: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(audioset_mdl_path, map_location=device)


frequncey stride=10, time stride=10
number of patches=1212


## 4. Linear Probing (Transfer Learning)
Freeze the transformer backbone and train only the MLP classifier head for the target domain.

In [ ]:
# Freezing backbone
for param in model.parameters():
    param.requires_grad = False
    
# Unfreezing MLP Head
for param in model.mlp_head.parameters():
    param.requires_grad = True

optimizer_probe = optim.AdamW(model.mlp_head.parameters(), lr=LR_PROBE)
best_val_acc = 0.0
best_checkpoint_path = os.path.join(CHECKPOINTS_DIR, f"best_ast_probe_{DATASET_TYPE}.pt")

print("Starting Linear Probing...")
for epoch in range(EPOCHS_PROBE):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch_idx, data in enumerate(train_loader):
        if data is None: continue
        inputs, labels, _, _, _ = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_probe.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct_train += preds.eq(labels).sum().item()
        total_train += labels.size(0)

    train_loss = running_loss / max(total_train, 1)
    train_acc = (correct_train / max(total_train, 1)) * 100

    # Validation phase
    model.eval()
    val_loss_total = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for batch_idx, data in enumerate(val_loader):
            if data is None: continue
            inputs, labels, _, _, _ = data
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss_total += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct_val += preds.eq(labels).sum().item()
            total_val += labels.size(0)
            
    val_loss = val_loss_total / max(total_val, 1)
    val_acc = (correct_val / max(total_val, 1)) * 100
    
    print(f"Probe Epoch [{epoch+1}/{EPOCHS_PROBE}] | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"  -> Best model saved (Val Acc: {val_acc:.2f}%)")

Starting Linear Probing...


c:\Users\caduo\dev\tcc\tcc\.venv\Lib\site-packages\torch\optim\adamw.py:529: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)
